# Step 3 — LSTM Sentiment Model

Project 2 (HPDP, SECP3133) — Real-Time Sentiment Analysis on Grab Google Play Reviews

**Owner:** Syahmi (NLP & Model Engineer) · **Model category:** Deep Learning (§6.3)

Architecture: **Embedding → LSTM → Dense → Softmax(3)**.

Why LSTM over Naive Bayes:
- Learns word *order* and context ("not good" vs "good"), which bag-of-words TF-IDF cannot
- Embeddings capture semantic similarity between words

Same severe class imbalance (Neutral ~5%) handled via **Keras `class_weight`**.
Uses the **same train/val/test split** from Step 1 so results are comparable to Naive Bayes.

> **Runtime:** enable GPU in Colab — `Runtime -> Change runtime type -> T4 GPU`. CPU works but is ~10x slower.

## 0. Setup + load splits

Hyperparameters chosen from Step 1's EDA: token p95 = 42, so `MAXLEN = 50` truncates almost nothing;
vocabulary ~31k unique words, so `VOCAB = 20000` keeps the common ones and sends the rare tail to `<OOV>`.

In [ ]:
import os, time, pickle, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, accuracy_score)

# Hyperparameters
VOCAB   = 20000
MAXLEN  = 50
EMB_DIM = 128
SEED    = 42
tf.random.set_seed(SEED); np.random.seed(SEED)

LABELS = ['Negative', 'Neutral', 'Positive']
print('GPU available:', tf.config.list_physical_devices('GPU') or 'NO (CPU only - slower)')

def find_file(stem):
    """Locate train/val/test, csv or xlsx. Includes Colab root (/content)."""
    roots = ['.', 'data', '/content', '/content/data', '../data']
    for r in roots:
        for ext in ('csv', 'xlsx'):
            p = f'{r}/{stem}.{ext}'
            if os.path.exists(p):
                return p
    raise FileNotFoundError(
        f'{stem}.(csv|xlsx) not found. On Colab, upload {stem}.csv to the file '
        'panel (it lands in /content/), then re-run this cell.')

def load(stem):
    p = find_file(stem)
    df = pd.read_excel(p) if p.endswith('.xlsx') else pd.read_csv(p)
    df['final_clean_text'] = df['final_clean_text'].fillna('')
    print(f'  {stem}: {len(df):,} rows  <- {p}')
    return df

MODELS_DIR = 'models'; os.makedirs(MODELS_DIR, exist_ok=True)
print('Loading splits...')
train = load('train'); val = load('val'); test = load('test')

## 1. Tokenize + pad sequences

Fit the tokenizer on **train only**. Convert each review to a sequence of word-IDs, then pad/truncate
to `MAXLEN`. The tokenizer is saved later — inference must use the exact same word->ID mapping.

In [ ]:
tokenizer = Tokenizer(num_words=VOCAB, oov_token='<OOV>')
tokenizer.fit_on_texts(train['final_clean_text'])

def encode(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAXLEN, padding='post', truncating='post')

X_train = encode(train['final_clean_text'])
X_val   = encode(val['final_clean_text'])
X_test  = encode(test['final_clean_text'])
y_train = train['sentiment_label'].astype(int).values
y_val   = val['sentiment_label'].astype(int).values
y_test  = test['sentiment_label'].astype(int).values

print(f'X_train: {X_train.shape}')
print(f'X_val:   {X_val.shape}')
print(f'X_test:  {X_test.shape}')

## 2. Class weights

Up-weight the rare Neutral class so the loss function doesn't let the model ignore it.

In [ ]:
weights = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=y_train)
class_weight = dict(enumerate(weights))
print('Class weights:', {LABELS[k]: round(v, 2) for k, v in class_weight.items()})

## 3. Build the model

In [ ]:
model = Sequential([
    Embedding(VOCAB, EMB_DIM, input_length=MAXLEN),
    SpatialDropout1D(0.3),
    LSTM(64, dropout=0.3, recurrent_dropout=0.0),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax'),
])
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.build(input_shape=(None, MAXLEN))
model.summary()

## 4. Train

Early stopping monitors **validation loss** and restores the best weights, so we can set a high
epoch ceiling without overfitting.

In [ ]:
es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

t0 = time.time()
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15, batch_size=128,
    class_weight=class_weight,
    callbacks=[es], verbose=1,
)
train_time = time.time() - t0
print(f'\nTrained in {train_time:.1f} s over {len(history.history["loss"])} epochs')

## 5. Training curves

Check for overfitting: train and val loss should track together. A growing gap = overfitting.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(history.history['loss'], label='train'); a1.plot(history.history['val_loss'], label='val')
a1.set_title('Loss'); a1.set_xlabel('Epoch'); a1.legend()
a2.plot(history.history['accuracy'], label='train'); a2.plot(history.history['val_accuracy'], label='val')
a2.set_title('Accuracy'); a2.set_xlabel('Epoch'); a2.legend()
plt.tight_layout(); plt.savefig('lstm_training_curves.png', dpi=120); plt.show()

## 6. Evaluate on the test set

In [ ]:
t0 = time.time()
probs = model.predict(X_test, batch_size=256, verbose=0)
infer_ms = (time.time() - t0) / len(y_test) * 1000
y_pred = probs.argmax(axis=1)

acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')
print(f'Accuracy : {acc:.4f}')
print(f'Macro-F1 : {macro_f1:.4f}')
print(f'Inference: {infer_ms:.4f} ms/sample\n')
print(classification_report(y_test, y_pred, target_names=LABELS, digits=4))

## 7. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=LABELS, yticklabels=LABELS, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'LSTM — Confusion Matrix\nAccuracy={acc:.3f}  Macro-F1={macro_f1:.3f}')
plt.tight_layout(); plt.savefig('lstm_confusion_matrix.png', dpi=120); plt.show()

## 8. Save model + tokenizer + metrics

In [ ]:
model.save(f'{MODELS_DIR}/lstm_model.keras')
with open(f'{MODELS_DIR}/lstm_tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
# Save the maxlen so inference pads identically
with open(f'{MODELS_DIR}/lstm_config.json', 'w') as f:
    json.dump({'maxlen': MAXLEN, 'vocab': VOCAB}, f)

report = classification_report(y_test, y_pred, target_names=LABELS, output_dict=True)
metrics = {
    'model': 'LSTM',
    'accuracy': acc, 'macro_f1': macro_f1,
    'macro_precision': report['macro avg']['precision'],
    'macro_recall': report['macro avg']['recall'],
    'f1_negative': report['Negative']['f1-score'],
    'f1_neutral': report['Neutral']['f1-score'],
    'f1_positive': report['Positive']['f1-score'],
    'train_time_s': train_time,
    'inference_ms_per_sample': infer_ms,
}
with open(f'{MODELS_DIR}/metrics_lstm.json', 'w') as f:
    json.dump(metrics, f, indent=2)

size_mb = os.path.getsize(f'{MODELS_DIR}/lstm_model.keras') / 1e6
print(f'Saved lstm_model.keras ({size_mb:.1f} MB) + tokenizer + config + metrics')
print('\n[OK] Step 3 complete.')

## 9. (Colab) Download outputs to your computer

In [ ]:
from google.colab import files

for f in ['lstm_model.keras', 'lstm_tokenizer.pkl', 'lstm_config.json', 'metrics_lstm.json']:
    files.download(f'{MODELS_DIR}/{f}')
for f in ['lstm_training_curves.png', 'lstm_confusion_matrix.png']:
    files.download(f)